In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from scipy.stats import rankdata # <--- Senjata Rahasia

base_path = '/kaggle/input/playground-series-s5e12'
train = pd.read_csv(os.path.join(base_path, 'train.csv'))
test = pd.read_csv(os.path.join(base_path, 'test.csv'))

# Feature Engineering (Yg sudah terbukti bagus)
def engineer_features(df):
    df = df.copy()
    df['pulse_pressure'] = df['systolic_bp'] - df['diastolic_bp']
    df['map'] = (df['systolic_bp'] + (2 * df['diastolic_bp'])) / 3
    df['chol_ratio'] = df['cholesterol_total'] / (df['hdl_cholesterol'] + 0.001)
    df['non_hdl'] = df['cholesterol_total'] - df['hdl_cholesterol']
    df['bmi_age_risk'] = df['bmi'] * df['age']
    df['visceral_fat_proxy'] = df['waist_to_hip_ratio'] * df['bmi']
    df['log_triglycerides'] = np.log1p(df['triglycerides'])
    return df

print("⚙️ Memproses Fitur...")
train_eng = engineer_features(train)
test_eng = engineer_features(test)

X = train_eng.drop(['id', 'diagnosed_diabetes'], axis=1)
y = train_eng['diagnosed_diabetes']
X_test = test_eng.drop(['id'], axis=1)

# Persiapan Kategori (Penting untuk masing-masing model)
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

# Data untuk CatBoost
X_cat = X.copy()
X_test_cat = X_test.copy()
X_cat[cat_cols] = X_cat[cat_cols].fillna("Missing")
X_test_cat[cat_cols] = X_test_cat[cat_cols].fillna("Missing")

# Data untuk LGBM & XGB (Perlu ubah jadi category code)
X_enc = X.copy()
X_test_enc = X_test.copy()
for col in cat_cols:
    X_enc[col] = X_enc[col].astype('category')
    X_test_enc[col] = X_test_enc[col].astype('category')

# --- 2. TRAINING 3 RAKSASA (FULL DATA) ---
# Kita tidak pakai CV Loop biar cepat, kita percaya pada parameter yang sudah kuat
# Kita pakai GPU untuk ketiganya

print("🔥 TRAINING 3 MODEL (MODE GPU)...")

# A. CATBOOST (High Precision)
print("1. CatBoost...", end=" ")
model_cat = CatBoostClassifier(
    iterations=1500, learning_rate=0.02, depth=6,
    border_count=254,       # Presisi tinggi
    l2_leaf_reg=5,
    random_seed=42, verbose=0, 
    task_type="GPU", devices='0',
    cat_features=cat_cols
)
model_cat.fit(X_cat, y)
pred_cat = model_cat.predict_proba(X_test_cat)[:, 1]
print("✅ Done.")

# B. LIGHTGBM
print("2. LightGBM...", end=" ")
model_lgbm = LGBMClassifier(
    n_estimators=1500, learning_rate=0.02, max_depth=5, num_leaves=31,
    random_state=42, verbose=-1, 
    device='gpu', gpu_platform_id=0, gpu_device_id=0
)
model_lgbm.fit(X_enc, y)
pred_lgbm = model_lgbm.predict_proba(X_test_enc)[:, 1]
print("✅ Done.")

# C. XGBOOST (Settingan Baru Anti-Error)
print("3. XGBoost...", end=" ")
model_xgb = XGBClassifier(
    n_estimators=1500, learning_rate=0.02, max_depth=5,
    tree_method='hist', device='cuda', # Settingan GPU Baru
    enable_categorical=True,           # Support kategori otomatis
    eval_metric='logloss',
    random_state=42
)
model_xgb.fit(X_enc, y)
pred_xgb = model_xgb.predict_proba(X_test_enc)[:, 1]
print("✅ Done.")

# --- 3. RANK AVERAGING ---
print("\n⚗️ Menghitung Ranking...")

# Ubah probabilitas jadi ranking (0 sampai 1)
rank_cat = rankdata(pred_cat) / len(pred_cat)
rank_lgbm = rankdata(pred_lgbm) / len(pred_lgbm)
rank_xgb = rankdata(pred_xgb) / len(pred_xgb)

# Gabung: Kita percaya CatBoost & LGBM lebih tinggi daripada XGB
# Bobot: Cat(40%) + LGBM(40%) + XGB(20%)
final_rank = (0.4 * rank_cat) + (0.4 * rank_lgbm) + (0.2 * rank_xgb)

submission = pd.DataFrame({'id': test['id'], 'diagnosed_diabetes': final_rank})
filename = 'submission_RANK_ENSEMBLE.csv'
submission.to_csv(filename, index=False)

print(f"\n🚀 SELESAI! Download file '{filename}'.")
print("Ini adalah kombinasi Peringkat dari 3 Model.")

/usr/local/lib/python3.12/dist-packages/sqlalchemy/orm/query.py:195: SyntaxWarning: "is not" with 'tuple' literal. Did you mean "!="?
  if entities is not ():


⚙️ Memproses Fitur...
🔥 TRAINING 3 MODEL (MODE GPU)...
1. CatBoost... ✅ Done.
2. LightGBM... 

1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


✅ Done.
3. XGBoost... 

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:774: UserWarning: [15:04:53] WARNING: /workspace/src/common/error_msg.cc:41: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


✅ Done.

⚗️ Menghitung Ranking...

🚀 SELESAI! Download file 'submission_RANK_ENSEMBLE.csv'.
Ini adalah kombinasi Peringkat dari 3 Model.
